# Behavior-Aware SASRec

Train and compare:
- **purchase_only**: SASRec on shared purchase sequences
- **behavior_aware**: same backbone + cart add/remove behavior embeddings

Aligned with Step1/Step2 shared evaluation:
- same `MAX_SEQ_LEN` / 14-day val / 14-day test cutoffs
- same `item_to_idx`
- same saved candidate sets (100 negatives + 1 target, **original product IDs**)
- metrics via `evaluation_utils.py`

Training follows the standard SASRec recipe (Kang & McAuley): left-padded sequences, dense next-item BCE at every non-pad timestep, predict from the **last** timestep at eval.


## 0. Setup


In [ ]:
from __future__ import annotations

from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence
import json
import pickle

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from evaluation_utils import evaluate_model_dataset

ARTIFACT_DIR = Path("artifacts")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch:", torch.__version__, "| device:", DEVICE)
print("Artifact dir:", ARTIFACT_DIR.resolve())


## 1. Config

`USE_SHARED_ARTIFACTS=True` loads Step1/Step2 pickles.  
`False` builds a tiny synthetic shared-style split for dry-runs.


In [ ]:
USE_SHARED_ARTIFACTS = True
MODE = "both"                 # "purchase_only" | "behavior_aware" | "both"
EVAL_SPLIT = "val"            # use "test" only for final numbers

DATA_CANDIDATES = [
    Path("data/subsampled"),
    Path("/content/subsampled"),
    Path("Desktop/DS5500/data/subsampled"),
    Path.home() / "Desktop" / "DS5500" / "data" / "subsampled",
]

HIDDEN_SIZE = 64
N_BLOCKS = 2
N_HEADS = 1
DROPOUT = 0.2
BATCH_SIZE = 128
LR = 1e-3
EPOCHS = 5 if not USE_SHARED_ARTIFACTS else 50
N_NEGATIVES_TRAIN = 1         # per timestep, matching SASRec
PATIENCE = 5
SEED = 42
# Optional: limit users while debugging (None = all)
MAX_TRAIN_USERS = None

OUTPUT_DIR = ARTIFACT_DIR / (
    "sasrec_shared_synthetic" if not USE_SHARED_ARTIFACTS else "sasrec_shared_run"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("USE_SHARED_ARTIFACTS =", USE_SHARED_ARTIFACTS)
print("OUTPUT_DIR =", OUTPUT_DIR)


## 2. Load shared sequences, vocabulary, candidates


In [ ]:
def build_synthetic_shared_artifacts(n_users=120, n_items=300, seed=42):
    rng = np.random.default_rng(seed)
    all_items = list(range(1, n_items + 1))
    item_to_idx = {sku: i + 1 for i, sku in enumerate(all_items)}
    idx_to_item = {i: sku for sku, i in item_to_idx.items()}

    train_sequences, val_targets, test_targets = {}, {}, {}
    for u in range(1, n_users + 1):
        L = int(rng.integers(2, 16))
        seq = rng.choice(all_items, size=L, replace=True).tolist()
        train_sequences[u] = seq
        val_targets[u] = int(rng.choice(all_items))
        test_targets[u] = int(rng.choice(all_items))

    val_eval_users = list(train_sequences)
    test_eval_users = list(train_sequences)

    def sample_cands(users, targets, seed_offset):
        local = np.random.default_rng(seed + seed_offset)
        out = {}
        pool = np.asarray(all_items)
        for u in users:
            target = targets[u]
            negs = local.choice(pool, size=100, replace=False).tolist()
            if target in negs:
                negs = [x for x in negs if x != target]
                while len(negs) < 100:
                    x = int(local.choice(pool))
                    if x != target and x not in negs:
                        negs.append(x)
            cands = negs[:100] + [target]
            local.shuffle(cands)
            out[u] = cands
        return out

    sequences = {
        "config": {
            "MAX_SEQ_LEN": 50,
            "TEST_DAYS": 14,
            "VAL_DAYS": 14,
            "val_cutoff": pd.Timestamp("2022-11-10"),
            "test_cutoff": pd.Timestamp("2022-11-24"),
        },
        "train_sequences": train_sequences,
        "val_targets": val_targets,
        "test_targets": test_targets,
        "val_eval_users": val_eval_users,
        "test_eval_users": test_eval_users,
        "item_to_idx": item_to_idx,
        "idx_to_item": idx_to_item,
        "n_items": len(all_items),
        "all_items": all_items,
    }
    candidates = {
        "val_candidate_sets": sample_cands(val_eval_users, val_targets, 1),
        "test_candidate_sets": sample_cands(test_eval_users, test_targets, 2),
        "n_negatives": 100,
        "eval_k": 10,
        "random_seed": seed,
        "candidate_pool": "full_item_vocabulary",
        "include_unseen_targets": True,
        "max_seq_len": 50,
        "val_cutoff": sequences["config"]["val_cutoff"],
        "test_cutoff": sequences["config"]["test_cutoff"],
    }
    return sequences, candidates


if USE_SHARED_ARTIFACTS:
    seq_path = ARTIFACT_DIR / "sequences_and_splits.pkl"
    cand_path = ARTIFACT_DIR / "shared_evaluation_candidates.pkl"
    if not seq_path.exists() or not cand_path.exists():
        raise FileNotFoundError(
            "Missing shared artifacts. Run Step1 and Step2 first:\n"
            f"  {seq_path}\n  {cand_path}"
        )
    with seq_path.open("rb") as f:
        shared = pickle.load(f)
    with cand_path.open("rb") as f:
        shared_cands = pickle.load(f)
    print("Loaded shared Step1/Step2 artifacts")
else:
    shared, shared_cands = build_synthetic_shared_artifacts(seed=SEED)
    print("Built synthetic shared-style artifacts")

train_sequences = shared["train_sequences"]
val_targets = shared["val_targets"]
test_targets = shared["test_targets"]
val_eval_users = list(shared["val_eval_users"])
test_eval_users = list(shared["test_eval_users"])
item_to_idx = shared["item_to_idx"]
idx_to_item = shared["idx_to_item"]
all_items = shared["all_items"]
n_items = int(shared["n_items"])
config = shared["config"]

MAX_SEQ_LEN = int(config["MAX_SEQ_LEN"])
VAL_CUTOFF = pd.Timestamp(config["val_cutoff"])
TEST_CUTOFF = pd.Timestamp(config["test_cutoff"])
EVAL_K = int(shared_cands.get("eval_k", 10))
N_NEGATIVES_EVAL = int(shared_cands.get("n_negatives", 100))
val_candidate_sets = shared_cands["val_candidate_sets"]
test_candidate_sets = shared_cands["test_candidate_sets"]

print(f"MAX_SEQ_LEN={MAX_SEQ_LEN} | VAL_CUTOFF={VAL_CUTOFF} | TEST_CUTOFF={TEST_CUTOFF}")
print(f"n_items={n_items:,} | train users={len(train_sequences):,}")
print(f"val_eval_users={len(val_eval_users):,} | test_eval_users={len(test_eval_users):,}")
print(f"candidates/user={N_NEGATIVES_EVAL + 1} | EVAL_K={EVAL_K}")

assert set(val_candidate_sets) == set(val_eval_users)
assert set(test_candidate_sets) == set(test_eval_users)
_u = val_eval_users[0]
assert len(val_candidate_sets[_u]) == N_NEGATIVES_EVAL + 1
assert val_targets[_u] in val_candidate_sets[_u]
print("Shared candidate checks passed")


## 3. Sanity check: popularity on shared candidates

This must match Step2 (~HitRate@10 **0.3882** on validation). If it does not, stop and fix artifact paths before training SASRec.


In [ ]:
item_popularity = Counter(
    item for sequence in train_sequences.values() for item in sequence
)

pop_scores = {
    user: [item_popularity.get(item, 0) for item in val_candidate_sets[user]]
    for user in val_eval_users
}
pop_summary, pop_results = evaluate_model_dataset(
    candidate_sets=val_candidate_sets,
    model_scores=pop_scores,
    targets={u: val_targets[u] for u in val_eval_users},
    k=EVAL_K,
)
print("Popularity baseline on shared val candidates:")
display(pd.DataFrame([pop_summary]))
print("Warm/cold for popularity:")
train_items = set(item_popularity)
pop_results = pop_results.copy()
pop_results["warm"] = pop_results["target_item"].isin(train_items)
display(
    pop_results.groupby("warm").agg(
        n=("user_id", "count"),
        HitRate_at_10=("HitRate@10", "mean"),
        NDCG_at_10=("NDCG@10", "mean"),
        MRR=("RR", "mean"),
    )
)


## 4. Build encoded histories

- **purchase_only**: shared `train_sequences` mapped through `item_to_idx`
- **behavior_aware**: buy+cart events with `timestamp < val_cutoff`, same shared vocab


In [ ]:
BEHAVIOR_PAD = 0
BEHAVIOR_BUY = 1
BEHAVIOR_ADD = 2
BEHAVIOR_REMOVE = 3


def encode_purchase_sequences(train_sequences, item_to_idx, max_users=None):
    users = list(train_sequences.keys())
    if max_users is not None:
        users = users[:max_users]
    out = {}
    for user in users:
        ids = [item_to_idx[s] for s in train_sequences[user] if s in item_to_idx]
        if ids:
            out[int(user)] = ids
    return out


def find_data_dir():
    return next((p for p in DATA_CANDIDATES if p.exists()), None)


def load_multibehavior_sequences(data_dir, item_to_idx, val_cutoff, max_len, max_users=None):
    frames = []
    file_map = {
        "product_buy": ("product_buy_subsampled.parquet", BEHAVIOR_BUY),
        "add_to_cart": ("add_to_cart_subsampled.parquet", BEHAVIOR_ADD),
        "remove_from_cart": ("remove_from_cart_subsampled.parquet", BEHAVIOR_REMOVE),
    }
    for behavior, (fname, beh_id) in file_map.items():
        path = Path(data_dir) / fname
        if not path.exists():
            raise FileNotFoundError(path)
        df = pd.read_parquet(path).drop_duplicates().copy()
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
        df = df.dropna(subset=["timestamp", "client_id", "sku"])
        df = df[df["timestamp"] < val_cutoff]
        df = df[df["sku"].isin(item_to_idx)]
        df["item_id"] = df["sku"].map(item_to_idx).astype(np.int64)
        df["behavior_id"] = beh_id
        frames.append(df[["client_id", "item_id", "behavior_id", "timestamp"]])
    events = pd.concat(frames, ignore_index=True)
    events = events.sort_values(["client_id", "timestamp"]).reset_index(drop=True)

    allow = None
    if max_users is not None:
        allow = set(list(train_sequences.keys())[:max_users])

    seq_items, seq_behs = {}, {}
    for user, group in events.groupby("client_id", sort=False):
        user = int(user)
        if allow is not None and user not in allow:
            continue
        items = group["item_id"].astype(int).tolist()[-max_len:]
        behs = group["behavior_id"].astype(int).tolist()[-max_len:]
        if items:
            seq_items[user] = items
            seq_behs[user] = behs
    return seq_items, seq_behs


def synthesize_cart_sequences(purchase_seqs, seed=42):
    rng = np.random.default_rng(seed)
    seq_items, seq_behs = {}, {}
    for user, buys in purchase_seqs.items():
        items, behs = [], []
        for item in buys:
            if rng.random() < 0.7:
                items.append(item)
                behs.append(BEHAVIOR_ADD)
            items.append(item)
            behs.append(BEHAVIOR_BUY)
        seq_items[user] = items[-MAX_SEQ_LEN:]
        seq_behs[user] = behs[-MAX_SEQ_LEN:]
    return seq_items, seq_behs


purchase_seqs = encode_purchase_sequences(train_sequences, item_to_idx, MAX_TRAIN_USERS)
print(f"purchase_only sequences: {len(purchase_seqs):,}")

behavior_item_seqs = None
behavior_beh_seqs = None
if MODE in ("behavior_aware", "both"):
    if not USE_SHARED_ARTIFACTS:
        behavior_item_seqs, behavior_beh_seqs = synthesize_cart_sequences(purchase_seqs, SEED)
        print(f"behavior_aware sequences (synthetic cart): {len(behavior_item_seqs):,}")
    else:
        data_dir = find_data_dir()
        if data_dir is None:
            print("WARNING: no parquet data; behavior_aware will be skipped")
        else:
            print("Loading cart+buy from", data_dir.resolve())
            behavior_item_seqs, behavior_beh_seqs = load_multibehavior_sequences(
                data_dir, item_to_idx, VAL_CUTOFF, MAX_SEQ_LEN, MAX_TRAIN_USERS
            )
            # Keep users that have a purchase train sequence
            keep = set(purchase_seqs)
            behavior_item_seqs = {u: s for u, s in behavior_item_seqs.items() if u in keep}
            behavior_beh_seqs = {u: behavior_beh_seqs[u] for u in behavior_item_seqs}
            print(f"behavior_aware sequences: {len(behavior_item_seqs):,}")


## 5. SASRec model + dense next-item dataset


In [ ]:
class PointWiseFeedForward(nn.Module):
    def __init__(self, hidden_size: int, dropout: float) -> None:
        super().__init__()
        self.conv1 = nn.Conv1d(hidden_size, hidden_size, kernel_size=1)
        self.dropout1 = nn.Dropout(dropout)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(hidden_size, hidden_size, kernel_size=1)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = x.transpose(1, 2)
        out = self.dropout2(self.conv2(self.relu(self.dropout1(self.conv1(out)))))
        return out.transpose(1, 2)


class SASRecBlock(nn.Module):
    def __init__(self, hidden_size: int, n_heads: int, dropout: float) -> None:
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_size, num_heads=n_heads, dropout=dropout, batch_first=True
        )
        self.ffn = PointWiseFeedForward(hidden_size, dropout)
        self.norm1 = nn.LayerNorm(hidden_size, eps=1e-8)
        self.norm2 = nn.LayerNorm(hidden_size, eps=1e-8)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attn_mask, key_padding_mask=None):
        h = self.norm1(x)
        attn_out, _ = self.attn(
            h, h, h, attn_mask=attn_mask, key_padding_mask=key_padding_mask, need_weights=False
        )
        x = x + self.dropout(attn_out)
        h = self.norm2(x)
        return x + self.ffn(h)


class SASRec(nn.Module):
    """SASRec closely following Kang & McAuley / pmixer PyTorch port."""

    def __init__(self, n_items, hidden_size=64, max_len=50, n_blocks=2, n_heads=1, dropout=0.2):
        super().__init__()
        self.n_items = n_items
        self.hidden_size = hidden_size
        self.max_len = max_len
        self.item_emb = nn.Embedding(n_items + 1, hidden_size, padding_idx=0)
        # index 0 reserved for pads (positions are 1..max_len on real tokens)
        self.pos_emb = nn.Embedding(max_len + 1, hidden_size, padding_idx=0)
        self.emb_dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            [SASRecBlock(hidden_size, n_heads, dropout) for _ in range(n_blocks)]
        )
        self.final_norm = nn.LayerNorm(hidden_size, eps=1e-8)
        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, (nn.Linear, nn.Embedding, nn.Conv1d)):
                nn.init.xavier_uniform_(module.weight)
            if isinstance(module, nn.Embedding) and module.padding_idx is not None:
                with torch.no_grad():
                    module.weight[module.padding_idx].fill_(0)
            if isinstance(module, nn.LayerNorm):
                nn.init.zeros_(module.bias)
                nn.init.ones_(module.weight)

    def _causal_mask(self, seq_len, device):
        return torch.triu(torch.ones(seq_len, seq_len, device=device, dtype=torch.bool), diagonal=1)

    def encode(self, item_ids, behavior_ids=None):
        del behavior_ids
        bsz, seq_len = item_ids.shape
        # Official-style positions: 0 on pads, 1..L on real tokens from the left of the window
        positions = torch.arange(1, seq_len + 1, device=item_ids.device).unsqueeze(0).expand(bsz, -1)
        positions = positions * (item_ids != 0).long()
        x = self.item_emb(item_ids) * (self.hidden_size ** 0.5)
        x = x + self.pos_emb(positions)
        x = self.emb_dropout(x)
        pad_mask = item_ids == 0
        attn_mask = self._causal_mask(seq_len, item_ids.device)
        for block in self.blocks:
            x = block(x, attn_mask=attn_mask, key_padding_mask=pad_mask)
        return self.final_norm(x)

    def forward_logits(self, item_ids, pos_ids, neg_ids, behavior_ids=None):
        feats = self.encode(item_ids, behavior_ids)
        pos_embs = self.item_emb(pos_ids)
        neg_embs = self.item_emb(neg_ids)
        pos_logits = (feats * pos_embs).sum(dim=-1)
        neg_logits = (feats * neg_embs).sum(dim=-1)
        return pos_logits, neg_logits

    def predict(self, item_ids, behavior_ids=None, candidate_items=None):
        feats = self.encode(item_ids, behavior_ids)
        final = feats[:, -1, :]
        if candidate_items is None:
            return torch.matmul(final, self.item_emb.weight.transpose(0, 1))
        cand_emb = self.item_emb(candidate_items)
        return (cand_emb * final.unsqueeze(1)).sum(dim=-1)


class BehaviorAwareSASRec(SASRec):
    def __init__(self, n_items, n_behaviors=3, hidden_size=64, max_len=50, n_blocks=2, n_heads=1, dropout=0.2):
        super().__init__(n_items, hidden_size, max_len, n_blocks, n_heads, dropout)
        self.behavior_emb = nn.Embedding(n_behaviors + 1, hidden_size, padding_idx=0)
        nn.init.xavier_uniform_(self.behavior_emb.weight)
        with torch.no_grad():
            self.behavior_emb.weight[0].fill_(0)

    def encode(self, item_ids, behavior_ids=None):
        if behavior_ids is None:
            raise ValueError("BehaviorAwareSASRec requires behavior_ids")
        bsz, seq_len = item_ids.shape
        positions = torch.arange(1, seq_len + 1, device=item_ids.device).unsqueeze(0).expand(bsz, -1)
        positions = positions * (item_ids != 0).long()
        x = self.item_emb(item_ids) * (self.hidden_size ** 0.5)
        x = x + self.behavior_emb(behavior_ids) + self.pos_emb(positions)
        x = self.emb_dropout(x)
        pad_mask = item_ids == 0
        attn_mask = self._causal_mask(seq_len, item_ids.device)
        for block in self.blocks:
            x = block(x, attn_mask=attn_mask, key_padding_mask=pad_mask)
        return self.final_norm(x)


def random_neq(n_items, banned, rng):
    t = int(rng.integers(1, n_items + 1))
    while t in banned:
        t = int(rng.integers(1, n_items + 1))
    return t


class SASRecTrainDataset(Dataset):
    """One training row per user: left-padded seq/pos/neg (official SASRec sampler).

    If behavior sequences are provided, loss targets are kept only when the
    next event is a purchase (cart events remain in the input context).
    """

    def __init__(
        self,
        item_seqs: Dict[int, List[int]],
        n_items: int,
        max_len: int = 50,
        seed: int = 42,
        behavior_seqs: Optional[Dict[int, List[int]]] = None,
        purchase_targets_only: bool = False,
    ):
        self.item_seqs = item_seqs
        self.behavior_seqs = behavior_seqs
        self.n_items = n_items
        self.max_len = max_len
        self.rng = np.random.default_rng(seed)
        self.purchase_targets_only = purchase_targets_only
        self.users = []
        for u, s in item_seqs.items():
            if len(s) < 2:
                continue
            if purchase_targets_only and behavior_seqs is not None:
                beh = behavior_seqs[u]
                # Need at least one timestep whose *next* event is a purchase.
                if not any(beh[i + 1] == BEHAVIOR_BUY for i in range(len(beh) - 1)):
                    continue
            self.users.append(u)

    def __len__(self):
        return len(self.users)

    def __getitem__(self, index):
        user = self.users[index]
        ts = self.item_seqs[user]
        beh = self.behavior_seqs[user] if self.behavior_seqs is not None else [BEHAVIOR_BUY] * len(ts)

        seq = np.zeros(self.max_len, dtype=np.int64)
        pos = np.zeros(self.max_len, dtype=np.int64)
        neg = np.zeros(self.max_len, dtype=np.int64)
        beh_pad = np.zeros(self.max_len, dtype=np.int64)
        pos_beh = np.zeros(self.max_len, dtype=np.int64)

        nxt = ts[-1]
        nxt_beh = beh[-1]
        idx = self.max_len - 1
        banned = set(ts)
        for i in reversed(range(len(ts) - 1)):
            seq[idx] = ts[i]
            beh_pad[idx] = beh[i]
            pos[idx] = nxt
            pos_beh[idx] = nxt_beh
            neg[idx] = random_neq(self.n_items, banned, self.rng)
            nxt = ts[i]
            nxt_beh = beh[i]
            idx -= 1
            if idx == -1:
                break

        if self.purchase_targets_only:
            # Keep input context, but drop non-purchase prediction heads.
            non_buy = pos_beh != BEHAVIOR_BUY
            pos[non_buy] = 0
            neg[non_buy] = 0

        return {
            "item_ids": torch.tensor(seq, dtype=torch.long),
            "behavior_ids": torch.tensor(beh_pad, dtype=torch.long),
            "pos_ids": torch.tensor(pos, dtype=torch.long),
            "neg_ids": torch.tensor(neg, dtype=torch.long),
        }


def collate(batch):
    return {k: torch.stack([b[k] for b in batch], 0) for k in batch[0]}


def sasrec_loss(pos_logits, neg_logits, pos_ids):
    indices = pos_ids != 0
    if indices.sum() == 0:
        return torch.zeros((), device=pos_logits.device)
    loss = F.binary_cross_entropy_with_logits(
        pos_logits[indices], torch.ones_like(pos_logits[indices])
    ) + F.binary_cross_entropy_with_logits(
        neg_logits[indices], torch.zeros_like(neg_logits[indices])
    )
    return loss


@dataclass
class TrainConfig:
    hidden_size: int = 64
    max_len: int = 50
    n_blocks: int = 2
    n_heads: int = 1
    dropout: float = 0.2
    batch_size: int = 128
    lr: float = 1e-3
    epochs: int = 50
    seed: int = 42
    device: str = "cpu"
    patience: int = 5


def set_seed(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)


def build_model(mode: str, n_items: int, cfg: TrainConfig):
    kwargs = dict(
        n_items=n_items,
        hidden_size=cfg.hidden_size,
        max_len=cfg.max_len,
        n_blocks=cfg.n_blocks,
        n_heads=cfg.n_heads,
        dropout=cfg.dropout,
    )
    if mode == "purchase_only":
        return SASRec(**kwargs)
    if mode == "behavior_aware":
        return BehaviorAwareSASRec(n_behaviors=3, **kwargs)
    raise ValueError(mode)


def left_pad_ids(ids: List[int], max_len: int) -> List[int]:
    ids = ids[-max_len:]
    return [0] * (max_len - len(ids)) + ids


print("SASRec helpers ready")


## 6. Score shared candidates (original product IDs, preserved order)


In [ ]:
@torch.no_grad()
def score_shared_candidates(
    model,
    item_seqs,
    candidate_sets,
    item_to_idx,
    max_len,
    device,
    behavior_seqs=None,
    batch_size=256,
):
    model.eval()
    users = [u for u in candidate_sets if u in item_seqs and item_seqs[u]]
    scores_out = {}
    for start in tqdm(range(0, len(users), batch_size), desc="score candidates", leave=False):
        batch_users = users[start:start + batch_size]
        item_rows, beh_rows, cand_rows, keep = [], [], [], []
        for u in batch_users:
            cands = candidate_sets[u]
            try:
                cand_idx = [item_to_idx[c] for c in cands]
            except KeyError:
                continue
            items = left_pad_ids(item_seqs[u], max_len)
            if behavior_seqs is None:
                behs = left_pad_ids([BEHAVIOR_BUY] * len(item_seqs[u]), max_len)
            else:
                behs = left_pad_ids(behavior_seqs[u], max_len)
            item_rows.append(items)
            beh_rows.append(behs)
            cand_rows.append(cand_idx)
            keep.append(u)
        if not keep:
            continue
        item_ids = torch.tensor(item_rows, dtype=torch.long, device=device)
        behavior_ids = torch.tensor(beh_rows, dtype=torch.long, device=device)
        cand_items = torch.tensor(cand_rows, dtype=torch.long, device=device)
        batch_scores = model.predict(item_ids, behavior_ids, candidate_items=cand_items)
        for i, u in enumerate(keep):
            scores_out[u] = batch_scores[i].detach().cpu().tolist()
    return scores_out


def evaluate_on_shared_split(model, item_seqs, split="val", behavior_seqs=None):
    if split == "val":
        cands, targets, users = val_candidate_sets, val_targets, val_eval_users
    else:
        cands, targets, users = test_candidate_sets, test_targets, test_eval_users
    cands = {u: cands[u] for u in users if u in cands and u in item_seqs}
    model_scores = score_shared_candidates(
        model, item_seqs, cands, item_to_idx, MAX_SEQ_LEN, DEVICE, behavior_seqs=behavior_seqs
    )
    cands = {u: cands[u] for u in model_scores}
    targets_sub = {u: targets[u] for u in model_scores}
    return evaluate_model_dataset(cands, model_scores, targets_sub, k=EVAL_K)


def warm_cold_breakdown(user_results):
    df = user_results.copy()
    df["target_seen_in_train"] = df["target_item"].isin(train_items)
    rows = (
        df.groupby("target_seen_in_train")
        .agg(
            n_users=("user_id", "count"),
            HitRate_at_10=("HitRate@10", "mean"),
            NDCG_at_10=("NDCG@10", "mean"),
            MRR=("RR", "mean"),
        )
        .reset_index()
    )
    rows["target_group"] = rows["target_seen_in_train"].map(
        {True: "Seen in training (warm)", False: "Unseen in training (cold)"}
    )
    return rows[["target_group", "n_users", "HitRate_at_10", "NDCG_at_10", "MRR"]]


print("Shared scoring helpers ready")


## 7. Train


In [ ]:
cfg = TrainConfig(
    hidden_size=HIDDEN_SIZE,
    max_len=MAX_SEQ_LEN,
    n_blocks=N_BLOCKS,
    n_heads=N_HEADS,
    dropout=DROPOUT,
    batch_size=BATCH_SIZE,
    lr=LR,
    epochs=EPOCHS,
    seed=SEED,
    device=DEVICE,
    patience=PATIENCE,
)
print(cfg)


def train_one(mode, item_seqs, behavior_seqs=None):
    set_seed(cfg.seed)
    device = torch.device(cfg.device)
    model = build_model(mode, n_items, cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, betas=(0.9, 0.98))

    ds = SASRecTrainDataset(
        item_seqs,
        n_items=n_items,
        max_len=cfg.max_len,
        seed=cfg.seed,
        behavior_seqs=behavior_seqs,
        purchase_targets_only=(behavior_seqs is not None),
    )
    loader = DataLoader(ds, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate)
    print(f"[{mode}] train users with len>=2: {len(ds):,}")

    best_metric = -1.0
    best_path = OUTPUT_DIR / mode / f"{mode}_best.pt"
    best_path.parent.mkdir(parents=True, exist_ok=True)
    patience_left = cfg.patience

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        losses = []
        # Resample negatives each epoch by rebuilding dataset RNG state via new dataset seed
        if epoch > 1:
            ds.rng = np.random.default_rng(cfg.seed + epoch)
        for batch in tqdm(loader, desc=f"{mode} epoch {epoch}", leave=False):
            item_ids = batch["item_ids"].to(device)
            behavior_ids = batch["behavior_ids"].to(device)
            pos_ids = batch["pos_ids"].to(device)
            neg_ids = batch["neg_ids"].to(device)
            optimizer.zero_grad(set_to_none=True)
            pos_logits, neg_logits = model.forward_logits(
                item_ids, pos_ids, neg_ids, behavior_ids=behavior_ids
            )
            loss = sasrec_loss(pos_logits, neg_logits, pos_ids)
            loss.backward()
            optimizer.step()
            losses.append(float(loss.item()))

        summary, _ = evaluate_on_shared_split(
            model, item_seqs, split="val", behavior_seqs=behavior_seqs
        )
        mean_loss = float(np.mean(losses)) if losses else None
        print(
            f"[{mode}] epoch={epoch} loss={mean_loss:.4f} "
            f"val HitRate@10={summary['HitRate@10']:.4f} "
            f"NDCG@10={summary['NDCG@10']:.4f} MRR={summary['MRR']:.4f} "
            f"n={summary['n_users']}"
        )
        metric = summary["NDCG@10"]
        if metric > best_metric:
            best_metric = metric
            patience_left = cfg.patience
            torch.save(
                {
                    "model_state": model.state_dict(),
                    "mode": mode,
                    "n_items": n_items,
                    "config": asdict(cfg),
                    "val_summary": summary,
                },
                best_path,
            )
        else:
            patience_left -= 1
            if patience_left <= 0:
                print(f"[{mode}] early stopping at epoch {epoch}")
                break

    ckpt = torch.load(best_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    return model, best_path, ckpt.get("val_summary")


trained = {}
modes = ["purchase_only", "behavior_aware"] if MODE == "both" else [MODE]
for mode in modes:
    if mode == "purchase_only":
        print("\n=== Training purchase_only ===")
        model, path, best_val = train_one(mode, purchase_seqs, behavior_seqs=None)
        trained[mode] = {
            "model": model,
            "item_seqs": purchase_seqs,
            "behavior_seqs": None,
            "checkpoint": path,
            "best_val": best_val,
        }
    else:
        if behavior_item_seqs is None:
            print("Skipping behavior_aware (no cart histories)")
            continue
        print("\n=== Training behavior_aware ===")
        model, path, best_val = train_one(
            mode, behavior_item_seqs, behavior_seqs=behavior_beh_seqs
        )
        trained[mode] = {
            "model": model,
            "item_seqs": behavior_item_seqs,
            "behavior_seqs": behavior_beh_seqs,
            "checkpoint": path,
            "best_val": best_val,
        }

print("Trained:", list(trained.keys()))


## 8. Evaluate on shared candidates


In [ ]:
all_summaries = []
all_breakdowns = {}

# Include popularity row for side-by-side comparison on the same split
if EVAL_SPLIT == "val":
    all_summaries.append({"model": "popularity", **pop_summary})

for mode, payload in trained.items():
    print(f"\n=== {mode} @ {EVAL_SPLIT} ===")
    summary, user_results = evaluate_on_shared_split(
        payload["model"],
        payload["item_seqs"],
        split=EVAL_SPLIT,
        behavior_seqs=payload["behavior_seqs"],
    )
    all_summaries.append({"model": mode, **summary})
    print(pd.DataFrame([{"model": mode, **summary}]))
    breakdown = warm_cold_breakdown(user_results)
    all_breakdowns[mode] = breakdown
    print("Warm / cold:")
    display(breakdown)
    out_csv = OUTPUT_DIR / mode / f"{mode}_{EVAL_SPLIT}_user_results.csv"
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    # *.csv is gitignored; still useful locally
    try:
        user_results.to_csv(out_csv, index=False)
        print("Wrote", out_csv)
    except Exception as e:
        print("Could not write csv:", e)

comparison = pd.DataFrame(all_summaries).set_index("model")
print("\nComparison vs popularity (same candidates/users):")
display(comparison)

summary_path = OUTPUT_DIR / f"summary_{EVAL_SPLIT}.json"
with summary_path.open("w") as f:
    json.dump(
        {
            "summaries": all_summaries,
            "breakdowns": {m: df.to_dict(orient="records") for m, df in all_breakdowns.items()},
            "eval_split": EVAL_SPLIT,
            "max_seq_len": MAX_SEQ_LEN,
            "val_cutoff": str(VAL_CUTOFF),
            "test_cutoff": str(TEST_CUTOFF),
            "n_items": n_items,
            "n_negatives_eval": N_NEGATIVES_EVAL,
        },
        f,
        indent=2,
        default=float,
    )
print("Wrote", summary_path.resolve())


## Notes

- Do **not** regenerate candidates — load `artifacts/shared_evaluation_candidates.pkl`.
- Scores must follow each candidate list’s order; `evaluation_utils` ranks them.
- Popularity sanity cell should read ~`HitRate@10=0.3882` on val before trusting SASRec numbers.
- Training uses official SASRec dense next-item BCE (left-padded `seq/pos/neg`), not leave-one-out instance loss.
- Sequences are left-padded; inference uses `feats[:, -1]`.
- Prefer `EVAL_SPLIT="val"` while developing; reserve test for the final report.
- About 30% of val users have only 1 train purchase — sequence models get little personal signal there; popularity remains a strong baseline on uniform full-vocab negatives.
